# LeetCode 713 — Subarray Product Less Than K bằng Python

Link đề: https://leetcode.com/problems/subarray-product-less-than-k/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Prefix Sum, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: ví dụ 1-2 lấy từ đề; các ví dụ khác do notebook tự đặt (đáp án đã kiểm chứng bằng code độc lập), cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **mảng con liên tiếp** (contiguous subarray) có **tích** tất cả phần tử **nhỏ hơn hẳn** `k`.
2. Nắm các công cụ Python cần dùng: `math.prod`, chia nguyên `//`, `enumerate`, hai con trỏ `l`, `r`, `bisect.bisect_left`, logarit `math.log` và điểm yếu của số thực dấu phẩy động, đệ quy chia để trị.
3. Đi từ **duyệt mọi mảng con `O(n^2)`** -> **cửa sổ trượt (hai con trỏ) `O(n)`** -> **logarit + tổng tiền tố + tìm kiếm nhị phân `O(n log n)`** -> **chia để trị, đếm các mảng con cắt qua điểm giữa `O(n log n)`**.
4. Hiểu hai mấu chốt: (a) mọi phần tử đều `>= 1` nên tích **không giảm** khi mở rộng mảng con sang phải (tính đơn điệu), vì thế cửa sổ hợp lệ chỉ cần dịch con trỏ trái **một chiều**; (b) khi đã có cửa sổ hợp lệ `[l..r]` thì có đúng `r - l + 1` mảng con hợp lệ **kết thúc tại `r`**.
5. Tránh các lỗi kinh điển: quên trường hợp `k <= 1` (cửa sổ rỗng, con trỏ vượt biên), dùng `>` thay vì `>=`, đếm `r - l` thay vì `r - l + 1`, dùng `if` thay vì `while` khi thu hẹp, và so sánh logarit không chừa biên an toàn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Trả về **số mảng con liên tiếp** mà **tích** của mọi phần tử trong mảng con **nhỏ hơn hẳn** `k`.

**Ví dụ 1**

- Input: `nums = [10,5,2,6]`, `k = 100`
- Output: `8`
- Giải thích: 8 mảng con có tích nhỏ hơn 100 là: `[10]`, `[5]`, `[2]`, `[6]`, `[10,5]`, `[5,2]`, `[2,6]`, `[5,2,6]`. Mảng con `[10,5,2]` có tích đúng `100` nên **không** tính (phải nhỏ hơn hẳn `k`).

**Ví dụ 2**

- Input: `nums = [1,2,3]`, `k = 0`
- Output: `0`

**Ràng buộc:**

- `1 <= nums.length <= 3 * 10^4`
- `1 <= nums[i] <= 1000`
- `0 <= k <= 10^6`

Chú ý quan trọng:

- Mảng con là một **đoạn liên tiếp** (không được bỏ qua phần tử ở giữa). Các mảng con khác **vị trí** được đếm riêng dù có cùng giá trị (ví dụ `[1]` ở hai vị trí khác nhau tính hai lần).
- Vì `nums[i] >= 1`, mọi tích đều `>= 1`; nên nếu `k <= 1` thì **không** có mảng con nào hợp lệ (đáp án `0`).
- Số mảng con của mảng dài `n` là `n(n+1)/2` (tới khoảng `4,5 * 10^8`), nên **không thể** liệt kê từng mảng con ở mức tối đa.

In [1]:
from typing import List
from bisect import bisect_left
from math import prod, log
import random
import time
import sys

examples = [
    (([10, 5, 2, 6], 100), 8),
    (([1, 2, 3], 0), 0),
]
for (nums, k), expected in examples:
    n = len(nums)
    print(f"nums={nums}, k={k}: tổng số mảng con = {n * (n + 1) // 2} | kỳ vọng {expected}")

nums=[10, 5, 2, 6], k=100: tổng số mảng con = 10 | kỳ vọng 8
nums=[1, 2, 3], k=0: tổng số mảng con = 6 | kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Mảng con liên tiếp và cách đếm

Mảng con xác định bởi hai chỉ số `i <= j`: `nums[i..j]`. Mảng dài `n` có `n` mảng con bắt đầu ở `0`, `n-1` bắt đầu ở `1`, ..., tổng `n(n+1)/2`. Cách cắt lát trong Python: `nums[i:j+1]`.

In [2]:
nums = [10, 5, 2, 6]
subs = [nums[i:j + 1] for i in range(len(nums)) for j in range(i, len(nums))]
print(len(subs), "mảng con:")
print(subs)

10 mảng con:
[[10], [10, 5], [10, 5, 2], [10, 5, 2, 6], [5], [5, 2], [5, 2, 6], [2], [2, 6], [6]]


### 3.2. `math.prod` và tích

`prod(iterable)` nhân mọi phần tử (`prod([])` = `1`). Số nguyên Python **không tràn** (kích thước tùy ý), nhưng tích lớn thì tốn thời gian: `1000^30000` có khoảng 90.000 chữ số; vì vậy ta nên tránh tích khổng lồ (cửa sổ trượt và việc cắt sớm sẽ giữ tích `< k * 1000`).

In [3]:
print(prod([10, 5, 2, 6]), prod([]), prod([7]))
print(len(str(1000 ** 1000)), "chữ số của 1000^1000 (Python giới hạn đổi số rất lớn ra chuỗi ở 4300 chữ số)")
for sub in ([10, 5], [5, 2], [10, 5, 2]):
    print(sub, "tích =", prod(sub), "| < 100?", prod(sub) < 100)

600 1 7
3001 chữ số của 1000^1000 (Python giới hạn đổi số rất lớn ra chuỗi ở 4300 chữ số)
[10, 5] tích = 50 | < 100? True
[5, 2] tích = 10 | < 100? True
[10, 5, 2] tích = 100 | < 100? False


### 3.3. Tính đơn điệu khi mọi số `>= 1`

Vì mỗi phần tử `>= 1`, **nhân thêm** một phần tử **không làm tích nhỏ đi**. Hai hệ quả:

1. Cố định đầu trái `i`: khi kéo đuôi phải `j` sang phải, tích **không giảm**. Do đó các `j` hợp lệ tạo thành một **đoạn liền** từ `i` (nếu `nums[i]` hợp lệ), và khi gặp `j` không hợp lệ thì mọi `j` lớn hơn cũng không hợp lệ.
2. Cố định đuôi phải `j`: khi thu hẹp đầu trái (tăng `i`), tích **không tăng**. Nếu `[i..j]` hợp lệ thì `[i+1..j]` cũng hợp lệ.

In [4]:
nums = [10, 5, 2, 6]
for i in range(len(nums)):
    products = [prod(nums[i:j + 1]) for j in range(i, len(nums))]
    print(f"bắt đầu ở i={i}: tích khi kéo phải = {products} (không giảm)")

bắt đầu ở i=0: tích khi kéo phải = [10, 50, 100, 600] (không giảm)
bắt đầu ở i=1: tích khi kéo phải = [5, 10, 60] (không giảm)
bắt đầu ở i=2: tích khi kéo phải = [2, 12] (không giảm)
bắt đầu ở i=3: tích khi kéo phải = [6] (không giảm)


### 3.4. Hai con trỏ (cửa sổ trượt) và công thức `r - l + 1`

Duy trì cửa sổ `[l..r]` luôn **hợp lệ** (tích `< k`). Khi `[l..r]` hợp lệ, mọi mảng con kết thúc tại `r` và bắt đầu từ `l, l+1, ..., r` đều hợp lệ (tính đơn điệu 2), tức **`r - l + 1`** mảng con. Cộng dồn qua mọi `r` được đáp án.

In [5]:
l, r = 1, 3
print("các mảng con kết thúc ở r=3, bắt đầu từ l=1 đến r:")
nums = [10, 5, 2, 6]
for start in range(l, r + 1):
    print("  ", nums[start:r + 1])
print("số lượng =", r - l + 1)

các mảng con kết thúc ở r=3, bắt đầu từ l=1 đến r:
   [5, 2, 6]
   [2, 6]
   [6]
số lượng = 3


### 3.5. Chia nguyên `//` để "bỏ" một thừa số khỏi tích

Khi `p` là tích cửa sổ, bỏ phần tử `nums[l]` khỏi cửa sổ là `p //= nums[l]`. Vì `nums[l]` **chia hết** `p` nên `//` cho kết quả chính xác (dùng `/` sẽ ra số thực).

In [6]:
p = 10 * 5 * 2
p //= 10
print(p, type(p))
q = 100
print(q / 10, type(q / 10), q // 10)

10 <class 'int'>
10.0 <class 'float'> 10


### 3.6. Logarit: biến tích thành tổng

`log(a * b) = log(a) + log(b)`. Vì `log` tăng chặt, `a * b < k` tương đương `log(a) + log(b) < log(k)`. Nhờ đó ta chuyển tích thành **tổng tiền tố**. Hạn chế: `math.log` cho **số thực** có sai số làm tròn, nên so sánh `<` ở **ranh giới** (tích đúng bằng `k`) có thể sai (mục 7 cách 3 và mục 13).

In [7]:
print(log(10) + log(5), log(50))
print(log(10 * 10), log(10) + log(10), log(100) == log(10) + log(10))
print(0.1 + 0.2, 0.1 + 0.2 == 0.3)

3.9120230054281464 3.912023005428146
4.605170185988092 4.605170185988092 True
0.30000000000000004 False


### 3.7. `bisect_left`: tìm kiếm nhị phân trong mảng đã sắp xếp

`bisect_left(a, x, lo)` trả về vị trí **đầu tiên** `j >= lo` sao cho `a[j] >= x` (mảng `a` tăng dần). Dùng để "tìm phần tử đầu tiên không nhỏ hơn `x`" trong `O(log n)`.

In [8]:
a = [0, 2, 5, 5, 9]
print(bisect_left(a, 5), bisect_left(a, 6), bisect_left(a, 10), bisect_left(a, 5, 3))

2 4 5 3


### 3.8. Chia để trị: đếm các mảng con "cắt qua điểm giữa"

Cắt mảng tại `mid`. Mỗi mảng con thuộc đúng một trong ba loại: nằm trọn ở nửa trái, nằm trọn ở nửa phải, hoặc **cắt qua** `mid` (chứa cả `nums[mid-1]` và `nums[mid]`). Hai loại đầu giải đệ quy; loại thứ ba đếm trực tiếp bằng hai con trỏ. Ví dụ đếm số mảng con của `[1,2,3,4]` theo kiểu này: `4 * 5 / 2 = 10`.

In [9]:
def count_all(lo, hi):
    """Số mảng con của nums[lo:hi] theo ba loại."""
    if hi - lo <= 1:
        return hi - lo
    mid = (lo + hi) // 2
    cross = (mid - lo) * (hi - mid)           # chọn đầu trái bên trái, đuôi phải bên phải
    return count_all(lo, mid) + count_all(mid, hi) + cross

print(count_all(0, 4), count_all(0, 10), 10 * 11 // 2)

10 55 55


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`nums = [10, 5, 2, 6]`, `k = 100`. Liệt kê **mọi** 10 mảng con và tích:

| Mảng con | Tích | `< 100`? |
|---|---|---|
| `[10]` | 10 | có |
| `[5]` | 5 | có |
| `[2]` | 2 | có |
| `[6]` | 6 | có |
| `[10,5]` | 50 | có |
| `[5,2]` | 10 | có |
| `[2,6]` | 12 | có |
| `[10,5,2]` | **100** | không (bằng `k`) |
| `[5,2,6]` | 60 | có |
| `[10,5,2,6]` | 600 | không |

Có `8` mảng con hợp lệ. Quan sát hai điều:

1. Khi bắt đầu ở `10`, tích tăng `10, 50, 100, 600`: hợp lệ rồi **hết hợp lệ mãi**.
2. Khi đếm theo **đuôi phải**: đuôi `5` có `[5]`, `[10,5]` (2 mảng); đuôi `2` có `[2]`, `[5,2]` (2 mảng, vì `[10,5,2] = 100` đã loại); đuôi `6` có `[6]`, `[2,6]`, `[5,2,6]` (3 mảng). Tổng `1 + 2 + 2 + 3 = 8`.

Số mảng kết thúc tại mỗi vị trí chính là **kích thước cửa sổ hợp lệ dài nhất kết thúc ở đó**. Đó là ý tưởng của cửa sổ trượt.

## 5. Cách 1 — Brute Force / baseline: duyệt mọi mảng con (`O(n^2)`)

**Ý tưởng.** Với mỗi đầu trái `i`, kéo đuôi phải `j` sang phải, giữ tích `p` chạy (nhân thêm `nums[j]`), đếm cho tới khi `p >= k` rồi **dừng** (vì tích không giảm, các `j` sau cũng không hợp lệ).

(Phiên bản "ngây thơ" tính lại tích từ đầu cho mỗi `(i, j)` là `O(n^3)`: ta đã bỏ bằng cách giữ tích chạy.)

In [10]:
def count_brute(nums, k):
    n = len(nums)
    count = 0
    for i in range(n):
        p = 1
        for j in range(i, n):
            p *= nums[j]
            if p >= k:
                break
            count += 1
    return count

for (nums, k), expected in examples:
    print(nums, k, "->", count_brute(nums, k), "| kỳ vọng", expected)

[10, 5, 2, 6] 100 -> 8 | kỳ vọng 8
[1, 2, 3] 0 -> 0 | kỳ vọng 0


In [11]:
class Solution1:
    """Cách 1: hai vòng lặp, tích chạy, dừng sớm."""
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        n = len(nums)
        count = 0
        for i in range(n):
            p = 1
            for j in range(i, n):
                p *= nums[j]
                if p >= k:
                    break
                count += 1
        return count

for (nums, k), expected in examples:
    assert Solution1().numSubarrayProductLessThanK(nums, k) == expected
print("cách 1 qua các ví dụ")

cách 1 qua các ví dụ


**Độ phức tạp.** Xấu nhất `O(n^2)`: khi nhiều phần tử là `1` và `k` lớn thì vòng trong không dừng sớm (ví dụ `n = 3*10^4` toàn `1`, `k = 10^6`: gần `4,5 * 10^8` bước). Space `O(1)`.

**Nút thắt.** Ta tính lại tích cho các mảng con chồng lấn nhau: mảng con `[i..j]` và `[i+1..j]` chia sẻ gần hết phần tử. Đo thử để thấy thời gian tăng gấp bốn khi `n` gấp đôi:

In [12]:
print("n    | brute (ms)")
for n in [500, 1000, 2000, 4000]:
    nums, k = [1] * n, 10 ** 6
    t0 = time.perf_counter()
    r = count_brute(nums, k)
    print(f"{n:>4} | {(time.perf_counter() - t0) * 1000:>9.1f}   (kết quả {r} = n(n+1)/2 = {n * (n + 1) // 2})")

n    | brute (ms)
 500 |      11.5   (kết quả 125250 = n(n+1)/2 = 125250)
1000 |      24.1   (kết quả 500500 = n(n+1)/2 = 500500)


2000 |     103.2   (kết quả 2001000 = n(n+1)/2 = 2001000)


4000 |     396.0   (kết quả 8002000 = n(n+1)/2 = 8002000)


Với `n = 3 * 10^4` ta cần khoảng `(30000 / 4000)^2 ~ 56` lần thời gian của dòng cuối: quá chậm. Cần một cách mà mỗi phần tử chỉ được "đụng" `O(1)` lần.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét ở mục 3.3 cho phép **không bao giờ lùi con trỏ**:

1. Với mỗi đuôi phải `r`, đặt `l(r)` là đầu trái **nhỏ nhất** sao cho `[l..r]` hợp lệ. Khi `r` tăng, `l(r)` **không giảm** (thêm phần tử không làm tích nhỏ đi nên cửa sổ không thể dài ra về bên trái).
2. Số mảng con hợp lệ kết thúc ở `r` bằng `r - l(r) + 1`.

Do đó dùng hai con trỏ `l` và `r` cùng chạy từ trái sang phải: mỗi con trỏ chỉ tiến, tổng cộng `O(n)` bước.

Cùng ý tưởng "đếm theo đuôi phải" có thể thực hiện bằng:

- **Cách 2:** cửa sổ trượt, giữ tích `p` bằng nhân/chia (khuyên dùng).
- **Cách 3:** đổi tích thành tổng logarit (prefix sum) rồi **tìm kiếm nhị phân** `l(r)` hoặc `r(l)`.
- **Cách 4:** **chia để trị**: đếm các mảng con cắt qua điểm giữa bằng hai con trỏ.

## 7. Các cách tiếp theo

### Cách 2 — Cửa sổ trượt (hai con trỏ) - khuyên dùng

**Giải thích từng đoạn.**

1. **Chặn `k <= 1`**: mọi tích `>= 1 >= k` nên không mảng con nào hợp lệ -> trả `0` (đồng thời tránh cửa sổ rỗng làm con trỏ vượt biên).
2. Khởi tạo `p = 1` (tích cửa sổ rỗng), `l = 0`, `ans = 0`.
3. Với mỗi `r` (và `x = nums[r]`): `p *= x` (mở rộng sang phải).
4. **Thu hẹp:** `while p >= k:` chia `p` cho `nums[l]` rồi `l += 1` (bỏ đầu trái). Vì `k > 1` và `p` giảm dần về `1`, vòng lặp luôn dừng, cùng lắm `l = r + 1` (cửa sổ rỗng khi `x >= k`).
5. **Đếm:** cộng `r - l + 1` (có thể bằng `0` khi cửa sổ rỗng).

Demo từng bước trên ví dụ 1:

In [13]:
def demo_window(nums, k):
    if k <= 1:
        return 0
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        print(f"r={r} x={x}: sau khi nhân p = {p}", end="")
        while p >= k:
            p //= nums[l]
            l += 1
            print(f" -> p >= {k}: bỏ nums[{l - 1}]={nums[l - 1]}, p = {p}, l = {l}", end="")
        ans += r - l + 1
        print(f" | cửa sổ [{l}..{r}] -> cộng {r - l + 1}, ans = {ans}")
    return ans

print("đáp án:", demo_window([10, 5, 2, 6], 100))

r=0 x=10: sau khi nhân p = 10 | cửa sổ [0..0] -> cộng 1, ans = 1
r=1 x=5: sau khi nhân p = 50 | cửa sổ [0..1] -> cộng 2, ans = 3
r=2 x=2: sau khi nhân p = 100 -> p >= 100: bỏ nums[0]=10, p = 10, l = 1 | cửa sổ [1..2] -> cộng 2, ans = 5
r=3 x=6: sau khi nhân p = 60 | cửa sổ [1..3] -> cộng 3, ans = 8
đáp án: 8


**Code nộp LeetCode (cách 2) - cũng là code chính ở mục 9:**

In [14]:
class Solution2:
    """Cách 2: cửa sổ trượt."""
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        if k <= 1:
            return 0
        p, l, ans = 1, 0, 0
        for r, x in enumerate(nums):
            p *= x
            while p >= k:
                p //= nums[l]
                l += 1
            ans += r - l + 1
        return ans

for (nums, k), expected in examples:
    assert Solution2().numSubarrayProductLessThanK(nums, k) == expected
print("cách 2 qua các ví dụ")

cách 2 qua các ví dụ


**Độ phức tạp (cách 2).** Mỗi phần tử được **nhân vào** đúng một lần (khi `r` đi qua) và **chia ra** tối đa một lần (khi `l` đi qua): **Time `O(n)`**, **Space `O(1)`**. Tích `p` luôn `< k * 1000 <= 10^9` trước khi thu hẹp nên số nhỏ, phép nhân/chia nhanh.

### Cách 3 — Logarit + tổng tiền tố + tìm kiếm nhị phân

**Giải thích.**

- Đặt `pre[0] = 0`, `pre[t] = log(nums[0]) + ... + log(nums[t-1])`. Mỗi `log(nums[t]) >= 0` nên `pre` **không giảm** -> có thể tìm kiếm nhị phân.
- Tích `nums[i..j]` `= exp(pre[j+1] - pre[i])`. Điều kiện `tích < k` tương đương `pre[j+1] - pre[i] < log(k)`.
- Với mỗi đầu trái `i`, tìm `j'` = chỉ số đầu tiên (`j' > i`) mà `pre[j'] >= pre[i] + T`, trong đó `T` là ngưỡng; khi đó các đuôi `j = i .. j'-2` hợp lệ: **`j' - i - 1`** mảng con bắt đầu ở `i`.

**Ngưỡng `T` quan trọng:** dùng `T = log(k)` có thể sai ở ranh giới do sai số làm tròn (mục 13). Vì tích là **số nguyên**, `tích < k` tương đương `tích <= k - 1`, tức nằm **dưới** `k - 0.5` một khoảng cách `>= 0.5`. Chọn **`T = log(k - 0.5)`**: biên an toàn cỡ `0.5/k >= 5 * 10^-7` ở cả hai phía, lớn hơn rất nhiều sai số cộng dồn của số thực (cỡ `10^-9`).

In [15]:
def demo_log(nums, k):
    pre = [0.0]
    for x in nums:
        pre.append(pre[-1] + log(x))
    T = log(k - 0.5)
    print("pre =", [round(v, 4) for v in pre], "| T = log(k - 0.5) =", round(T, 4))
    ans = 0
    for i in range(len(nums)):
        j = bisect_left(pre, pre[i] + T, i + 1)       # chỉ số đầu tiên có pre[j] >= pre[i] + T
        print(f"i={i}: pre[i]+T = {pre[i] + T:.4f} -> j' = {j}, số mảng bắt đầu ở i = {j - i - 1}")
        ans += j - i - 1
    return ans

print("đáp án:", demo_log([10, 5, 2, 6], 100))

pre = [0.0, 2.3026, 3.912, 4.6052, 6.3969] | T = log(k - 0.5) = 4.6002
i=0: pre[i]+T = 4.6002 -> j' = 3, số mảng bắt đầu ở i = 2
i=1: pre[i]+T = 6.9027 -> j' = 5, số mảng bắt đầu ở i = 3
i=2: pre[i]+T = 8.5122 -> j' = 5, số mảng bắt đầu ở i = 2
i=3: pre[i]+T = 9.2053 -> j' = 5, số mảng bắt đầu ở i = 1
đáp án: 8


**Code nộp LeetCode (cách 3):**

In [16]:
class Solution3:
    """Cách 3: tổng logarit tiền tố + tìm kiếm nhị phân."""
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        if k <= 1:
            return 0
        pre = [0.0]
        for x in nums:
            pre.append(pre[-1] + log(x))
        T = log(k - 0.5)               # tích nguyên < k  <=>  tích < k - 0.5
        ans = 0
        for i in range(len(nums)):
            j = bisect_left(pre, pre[i] + T, i + 1)
            ans += j - i - 1
        return ans

for (nums, k), expected in examples:
    assert Solution3().numSubarrayProductLessThanK(nums, k) == expected
print("cách 3 qua các ví dụ")

cách 3 qua các ví dụ


**Độ phức tạp (cách 3).** Tính `pre` là `O(n)`; mỗi đầu trái một lần `bisect_left` `O(log n)`: **Time `O(n log n)`**, **Space `O(n)`** cho mảng `pre`. Chậm hơn cách 2 và dính rủi ro số thực; có giá trị học về kỹ thuật "tích -> tổng" và cho thấy mọi bài tích dương có thể đổi sang tổng.

### Cách 4 — Chia để trị: đếm các mảng con cắt qua điểm giữa

**Giải thích.**

- `solve(lo, hi)` đếm mảng con hợp lệ trong `nums[lo:hi]`: nếu `hi - lo == 1` thì `1` nếu `nums[lo] < k`, ngược lại `0`.
- Ngược lại cắt tại `mid`: `solve(lo, mid) + solve(mid, hi) + cross(lo, mid, hi)`.
- **`cross`:** một mảng con cắt qua `mid` gồm một hậu tố không rỗng của nửa trái `nums[a..mid-1]` và một tiền tố không rỗng của nửa phải `nums[mid..b]`. Tích `= L(a) * R(b)`.
  - Tính danh sách `Lp` (tích các hậu tố trái, dài tăng dần: `nums[mid-1]`, `nums[mid-2]*nums[mid-1]`, ...) và `Rp` (tích các tiền tố phải). Cả hai **không giảm**; ta **dừng** thêm phần tử ngay khi tích `>= k` (mọi tích lớn hơn đều vô ích), nên các giá trị luôn `< k`.
  - Với mỗi phần tử `Lp[a]` theo thứ tự **tăng dần**, số `b` thỏa `Lp[a] * Rp[b] < k` là một tiền tố của `Rp`, và độ dài tiền tố đó **không tăng** khi `Lp[a]` tăng. Dùng một con trỏ `j` chỉ giảm.

In [17]:
def cross_count(nums, lo, mid, hi, k):
    Lp, p = [], 1
    for t in range(mid - 1, lo - 1, -1):
        p *= nums[t]
        if p >= k:
            break
        Lp.append(p)
    Rp, p = [], 1
    for t in range(mid, hi):
        p *= nums[t]
        if p >= k:
            break
        Rp.append(p)
    total, j = 0, len(Rp)
    for a in Lp:
        while j > 0 and a * Rp[j - 1] >= k:
            j -= 1
        total += j
    return total, Lp, Rp

nums, k = [10, 5, 2, 6], 100
total, Lp, Rp = cross_count(nums, 0, 2, 4, k)
print("Lp (tích hậu tố của [10,5]) =", Lp, "| Rp (tích tiền tố của [2,6]) =", Rp, "| cặp thỏa =", total)
print("các mảng con cắt qua giữa: [5,2]=10, [5,2,6]=60, [10,5,2]=100 (loại), [10,5,2,6]=600 (loại) -> 2")

Lp (tích hậu tố của [10,5]) = [5, 50] | Rp (tích tiền tố của [2,6]) = [2, 12] | cặp thỏa = 2
các mảng con cắt qua giữa: [5,2]=10, [5,2,6]=60, [10,5,2]=100 (loại), [10,5,2,6]=600 (loại) -> 2


**Code nộp LeetCode (cách 4):**

In [18]:
class Solution4:
    """Cách 4: chia để trị, đếm riêng các mảng con cắt qua điểm giữa."""
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        if k <= 1:
            return 0

        def solve(lo, hi):
            if hi - lo == 1:
                return 1 if nums[lo] < k else 0
            mid = (lo + hi) // 2
            Lp, p = [], 1
            for t in range(mid - 1, lo - 1, -1):
                p *= nums[t]
                if p >= k:
                    break
                Lp.append(p)
            Rp, p = [], 1
            for t in range(mid, hi):
                p *= nums[t]
                if p >= k:
                    break
                Rp.append(p)
            cross, j = 0, len(Rp)
            for a in Lp:
                while j > 0 and a * Rp[j - 1] >= k:
                    j -= 1
                cross += j
            return solve(lo, mid) + solve(mid, hi) + cross

        return solve(0, len(nums))

for (nums, k), expected in examples:
    assert Solution4().numSubarrayProductLessThanK(nums, k) == expected
print("cách 4 qua các ví dụ")

cách 4 qua các ví dụ


**Độ phức tạp (cách 4).** Mỗi mức đệ quy tốn `O(n)` (xây `Lp`, `Rp` tối đa độ dài đoạn và quét hai con trỏ một lần), có `log n` mức: **Time `O(n log n)`**. Space: `O(n)` cho hai danh sách tạm (chỉ một cặp tại một thời điểm) và `O(log n)` ngăn xếp đệ quy. Cách này không phụ thuộc tính chất "đuôi phải" nên **dễ mở rộng** sang các bài phức tạp hơn (khi cửa sổ trượt không áp dụng được).

**Tóm tắt bốn cách:** (1) hai vòng lặp `O(n^2)`; (2) cửa sổ trượt `O(n)`; (3) logarit + tìm kiếm nhị phân `O(n log n)`; (4) chia để trị `O(n log n)`.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (cửa sổ trượt) trên `[10, 5, 2, 6]`, `k = 100`

Mỗi dòng ứng với một giá trị của `r`: tích sau khi nhân, các bước thu hẹp, cửa sổ, số mảng con cộng thêm và tổng.

| Cột | Ý nghĩa |
|---|---|
| `r`, `x` | đuôi phải hiện tại và giá trị |
| `p` | tích cửa sổ |
| `l` | đầu trái của cửa sổ hợp lệ |
| `+` | `r - l + 1` mảng con kết thúc tại `r` |

In [19]:
def trace_window(nums, k):
    if k <= 1:
        print("k <= 1 -> 0")
        return 0
    p, l, ans = 1, 0, 0
    print(f"{'r':>2} {'x':>4} | {'p sau nhân':>10} | thu hẹp | {'l':>2} {'p':>6} | {'+':>2} {'ans':>4}")
    for r, x in enumerate(nums):
        p *= x
        before = p
        steps = []
        while p >= k:
            steps.append(f"bỏ {nums[l]}")
            p //= nums[l]
            l += 1
        ans += r - l + 1
        print(f"{r:>2} {x:>4} | {before:>10} | {', '.join(steps) or '-':<10}| {l:>2} {p:>6} | {r - l + 1:>2} {ans:>4}")
    return ans

print("đáp án:", trace_window([10, 5, 2, 6], 100))

 r    x | p sau nhân | thu hẹp |  l      p |  +  ans
 0   10 |         10 | -         |  0     10 |  1    1
 1    5 |         50 | -         |  0     50 |  2    3
 2    2 |        100 | bỏ 10     |  1     10 |  2    5
 3    6 |         60 | -         |  1     60 |  3    8
đáp án: 8


Ở `r = 2` (`x = 2`): `p = 100 >= 100` nên bỏ `10`, `p = 10`, `l = 1`; cửa sổ `[5, 2]` có `2` mảng con. Đúng như bảng mục 4. Thử một ví dụ có phần tử đơn lẻ `>= k` (cửa sổ rỗng):

In [20]:
print("đáp án:", trace_window([3, 200, 2, 4], 50))

 r    x | p sau nhân | thu hẹp |  l      p |  +  ans
 0    3 |          3 | -         |  0      3 |  1    1
 1  200 |        600 | bỏ 3, bỏ 200|  2      1 |  0    1
 2    2 |          2 | -         |  2      2 |  1    2
 3    4 |          8 | -         |  2      8 |  2    4
đáp án: 4


Ở `r = 1` (`x = 200 >= 50`): cửa sổ co về rỗng (`l = 2 = r + 1`, `p = 1`), cộng `0`. Sau đó cửa sổ bắt đầu lại từ `r = 2`.

### 8.2. Trace cách 4 (chia để trị)

In từng đoạn `[lo, hi)`, điểm cắt, danh sách `Lp`, `Rp`, và số mảng con cắt qua.

In [21]:
def trace_dc(nums, k):
    def solve(lo, hi, depth):
        pad = "  " * depth
        if hi - lo == 1:
            r = 1 if nums[lo] < k else 0
            print(f"{pad}[{lo},{hi}) = {nums[lo:hi]}: một phần tử -> {r}")
            return r
        mid = (lo + hi) // 2
        cross, Lp, Rp = cross_count(nums, lo, mid, hi, k)
        print(f"{pad}[{lo},{hi}) = {nums[lo:hi]}, cắt ở {mid}: Lp={Lp}, Rp={Rp} -> cắt qua = {cross}")
        left = solve(lo, mid, depth + 1)
        right = solve(mid, hi, depth + 1)
        print(f"{pad}=> [{lo},{hi}) = {left} + {right} + {cross} = {left + right + cross}")
        return left + right + cross
    return solve(0, len(nums), 0)

print("đáp án:", trace_dc([10, 5, 2, 6], 100))

[0,4) = [10, 5, 2, 6], cắt ở 2: Lp=[5, 50], Rp=[2, 12] -> cắt qua = 2
  [0,2) = [10, 5], cắt ở 1: Lp=[10], Rp=[5] -> cắt qua = 1
    [0,1) = [10]: một phần tử -> 1
    [1,2) = [5]: một phần tử -> 1
  => [0,2) = 1 + 1 + 1 = 3
  [2,4) = [2, 6], cắt ở 3: Lp=[2], Rp=[6] -> cắt qua = 1
    [2,3) = [2]: một phần tử -> 1
    [3,4) = [6]: một phần tử -> 1
  => [2,4) = 1 + 1 + 1 = 3
=> [0,4) = 3 + 3 + 2 = 8
đáp án: 8


Tại gốc: nửa trái `[10,5]` có `3` mảng (`[10]`, `[5]`, `[10,5]`), nửa phải `[2,6]` có `3` mảng, và `2` mảng cắt qua (`[5,2]`, `[5,2,6]`), tổng `8`.

### 8.3. Trace cách 1 (hai vòng lặp) để thấy việc dừng sớm

In [22]:
def trace_brute(nums, k):
    count = 0
    for i in range(len(nums)):
        p, row = 1, []
        for j in range(i, len(nums)):
            p *= nums[j]
            if p >= k:
                row.append(f"p={p}>=k dừng")
                break
            count += 1
            row.append(f"[{i}..{j}]={p}")
        print(f"i={i}: " + ", ".join(row))
    return count

print("đáp án:", trace_brute([10, 5, 2, 6], 100))

i=0: [0..0]=10, [0..1]=50, p=100>=k dừng
i=1: [1..1]=5, [1..2]=10, [1..3]=60
i=2: [2..2]=2, [2..3]=12
i=3: [3..3]=6
đáp án: 8


## 9. Code nộp LeetCode

Dùng **cách 2** (cửa sổ trượt):

```python
class Solution:
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        if k <= 1:
            return 0
        p, l, ans = 1, 0, 0
        for r, x in enumerate(nums):
            p *= x
            while p >= k:
                p //= nums[l]
                l += 1
            ans += r - l + 1
        return ans
```

Chạy lại đúng đoạn code trên:

In [23]:
class Solution:
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        if k <= 1:
            return 0
        p, l, ans = 1, 0, 0
        for r, x in enumerate(nums):
            p *= x
            while p >= k:
                p //= nums[l]
                l += 1
            ans += r - l + 1
        return ans

# Mô phỏng cách LeetCode gọi
for (nums, k), expected in examples:
    got = Solution().numSubarrayProductLessThanK(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[10, 5, 2, 6] 100 -> 8 OK
[1, 2, 3] 0 -> 0 OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** Với mỗi `r`, gọi `c(r)` là số mảng con hợp lệ **kết thúc tại `r`**; đáp án là `sum c(r)`. Gọi `L(r)` là đầu trái nhỏ nhất sao cho `nums[L(r)..r]` có tích `< k` (nếu không tồn tại thì `L(r) = r + 1`).

**Tính chất 1 (đơn điệu theo đầu trái):** vì mọi `nums[i] >= 1`, nếu `nums[a..r]` hợp lệ thì với mọi `a <= b <= r` mảng `nums[b..r]` có tích **không lớn hơn** nên cũng hợp lệ. Do đó các đầu trái hợp lệ cho đuôi `r` là `L(r), L(r)+1, ..., r` và `c(r) = r - L(r) + 1`.

**Tính chất 2 (L không giảm theo r):** giả sử `L(r+1) < L(r)`. Khi đó `nums[L(r+1)..r+1]` hợp lệ, nên theo tính chất 1 (bỏ phần tử cuối cũng không làm tích tăng vì chia cho số `>= 1`), `nums[L(r+1)..r]` hợp lệ, mâu thuẫn với tính nhỏ nhất của `L(r)`. Vậy `L(r+1) >= L(r)`.

**Thuật toán duy trì `l = L(r)`:** vòng lặp bất biến "trước mỗi `r`, `p` là tích của `nums[l..r-1]`, nó `< k` (hoặc cửa sổ rỗng với `p = 1`), và `l = L(r-1)`". Nhân `nums[r]` ta được tích `nums[l..r]`. Nếu `>= k`, theo tính chất 1 ta phải tăng `l` cho tới khi tích `< k`: mỗi lần tăng `l` ta chia `p` cho `nums[l]` (chia hết, vì `nums[l]` đã nằm trong tích). Dừng tại giá trị `l` **nhỏ nhất** làm `p < k`, chính là `L(r)` (không dừng sớm: bước trước `p >= k`; không đi quá: dừng ngay lần đầu `p < k`). Vòng thu hẹp luôn kết thúc vì sau cùng `p = 1 < k` (do `k > 1`), khi `l = r + 1` (cửa sổ rỗng).

**Điều kiện `k <= 1`:** mọi tích `>= 1 >= k` nên không có mảng con nào, trả `0` (và tránh vòng `while` chạy mãi khiến `l` vượt chiều dài mảng).

**Cách 1** liệt kê trực tiếp và dừng sớm theo đơn điệu (cố định `i`, tích tăng theo `j`). **Cách 3**: `tích < k` tương đương `tổng log < log(k - 0.5)` vì tích nguyên và biên an toàn; `pre` không giảm nên `bisect_left` đúng. **Cách 4**: phân loại mỗi mảng con theo vị trí so với `mid` (bao trùm mọi trường hợp, không trùng lặp); với loại cắt qua, hai danh sách tích không giảm và quan hệ "số `b` hợp lệ giảm khi `Lp[a]` tăng" cho phép đếm bằng hai con trỏ.

## 11. Phân tích Time / Space Complexity

`n = len(nums)` (tối đa `3 * 10^4`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp, dừng sớm | `O(n^2)` xấu nhất | `O(1)` | chậm khi nhiều số `1` |
| 2. Cửa sổ trượt | `O(n)` | `O(1)` | **dùng để nộp** |
| 3. Log + nhị phân | `O(n log n)` | `O(n)` | rủi ro sai số float (đã chừa biên) |
| 4. Chia để trị | `O(n log n)` | `O(n)` tạm + `O(log n)` đệ quy | dễ mở rộng sang bài khó hơn |

Giá trị đáp án tối đa `n(n+1)/2 ~ 4,5 * 10^8`: Python không lo tràn. Tích `p` trong cách 2 luôn `< k * 1000 <= 10^9` ngay sau khi nhân, nhỏ.

## 12. Bộ test

### 12.1. Ví dụ của đề và oracle độc lập

Oracle độc lập: với mảng nhỏ, **tính tích từng mảng con bằng `math.prod` từ đầu** (không dùng tính đơn điệu, không dừng sớm). Chậm (`O(n^3)`) nhưng hiển nhiên đúng.

In [24]:
def oracle_prod(nums, k):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if prod(nums[i:j + 1]) < k)

METHODS = {
    "1-hai-vong": Solution1().numSubarrayProductLessThanK,
    "2-cua-so": Solution2().numSubarrayProductLessThanK,
    "3-log-nhi-phan": Solution3().numSubarrayProductLessThanK,
    "4-chia-de-tri": Solution4().numSubarrayProductLessThanK,
    "nop": Solution().numSubarrayProductLessThanK,
}

def check_all(nums, k, expected=None, skip=()):
    exp = oracle_prod(nums, k) if expected is None else expected
    for name, fn in METHODS.items():
        if name in skip:
            continue
        got = fn(list(nums), k)
        assert got == exp, (name, nums, k, got, exp)
    return exp

for (nums, k), expected in examples:
    assert check_all(nums, k, expected) == oracle_prod(nums, k) == expected
check_all([1, 2, 3], 4, expected=4)          # tính tay ở mục 19
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.2. Edge case

- `k = 0` và `k = 1`: luôn `0` (vì mọi tích `>= 1`).
- `k = 2`: chỉ các mảng con toàn số `1` hợp lệ.
- Mảng một phần tử, nhỏ hơn `k`, bằng `k`, lớn hơn `k`.
- Mảng toàn `1` (mọi mảng con hợp lệ khi `k >= 2`): đáp án `n(n+1)/2`.
- Có phần tử `>= k` ở giữa (cắt mảng thành hai khúc).
- Tích đúng bằng `k` (phải **không** tính).
- Giá trị lớn `1000` và `k = 10^6` (hai phần tử `1000` nhân ra đúng `10^6`, không tính).

In [25]:
edge = [
    ([1, 2, 3], 0), ([1, 2, 3], 1), ([1], 1), ([1], 2),
    ([5], 5), ([5], 6), ([7], 3),
    ([1, 1, 1], 2), ([1, 1, 1, 1], 1),
    ([2, 2, 2], 2), ([2, 2, 2], 4),
    ([3, 200, 2, 4], 50),
    ([1000, 1000], 10 ** 6), ([1000, 1000], 10 ** 6 + 1),
    ([10, 5, 2, 6], 100), ([10, 5, 2, 6], 101), ([10, 5, 2, 6], 600), ([10, 5, 2, 6], 601),
]
for nums, k in edge:
    exp = check_all(nums, k)
    print(f"nums={nums!s:20} k={k:<8} -> {exp}")
assert check_all([1] * 50, 10 ** 6) == 50 * 51 // 2
assert check_all([2, 2, 2], 4) == 3              # chỉ các phần tử đơn lẻ (2*2 = 4 không < 4)
assert check_all([3, 200, 2, 4], 50) == 4        # [3], [2], [4], [2,4]
assert check_all([1000, 1000], 10 ** 6) == 2     # 1000*1000 = k: không tính
assert check_all([1000, 1000], 10 ** 6 + 1) == 3
assert check_all([10, 5, 2, 6], 101) == 9        # [10,5,2] = 100 < 101 được tính thêm
assert check_all([10, 5, 2, 6], 600) == 9        # 600 không < 600
assert check_all([10, 5, 2, 6], 601) == 10
print("edge case: OK")

nums=[1, 2, 3]            k=0        -> 0
nums=[1, 2, 3]            k=1        -> 0
nums=[1]                  k=1        -> 0
nums=[1]                  k=2        -> 1
nums=[5]                  k=5        -> 0
nums=[5]                  k=6        -> 1
nums=[7]                  k=3        -> 0
nums=[1, 1, 1]            k=2        -> 6
nums=[1, 1, 1, 1]         k=1        -> 0
nums=[2, 2, 2]            k=2        -> 0
nums=[2, 2, 2]            k=4        -> 3
nums=[3, 200, 2, 4]       k=50       -> 4
nums=[1000, 1000]         k=1000000  -> 2
nums=[1000, 1000]         k=1000001  -> 3
nums=[10, 5, 2, 6]        k=100      -> 8
nums=[10, 5, 2, 6]        k=101      -> 9
nums=[10, 5, 2, 6]        k=600      -> 9
nums=[10, 5, 2, 6]        k=601      -> 10
edge case: OK


### 12.3. Phản ví dụ: input làm lộ các lỗi quen thuộc

- Tích **đúng bằng** `k` (kiểm `>=` so với `>`): `[10,5,2]` với `k=100`.
- Phần tử đơn lẻ **lớn hơn** `k` (cửa sổ phải co về rỗng).
- `k = 1` và `k = 0` (con trỏ vượt biên nếu thiếu chặn).
- Ranh giới của cách logarit: các tích **đúng bằng `k`** như `[2,4,3]` với `k = 12` (mục 13).

In [26]:
tricky = [
    ([10, 5, 2], 100),
    ([100, 1, 1], 50),
    ([1, 1, 1], 1),
    ([2, 4, 3], 12),
    ([3, 6, 8, 5, 9], 720),
    ([6, 6, 6, 6], 36),
    ([1, 1, 2, 1, 1], 2),
]
for nums, k in tricky:
    print(f"{nums!s:18} k={k:<4} -> {check_all(nums, k)}")

[10, 5, 2]         k=100  -> 5
[100, 1, 1]        k=50   -> 3
[1, 1, 1]          k=1    -> 0
[2, 4, 3]          k=12   -> 4
[3, 6, 8, 5, 9]    k=720  -> 12
[6, 6, 6, 6]       k=36   -> 4
[1, 1, 2, 1, 1]    k=2    -> 6


### 12.4. Đối chiếu ngẫu nhiên (seed cố định) với oracle

Mảng ngẫu nhiên độ dài `1..12`, giá trị `1..8` (nhiều mảng con có tích gần `k`), `k` từ `0` đến `300`. Cả năm cách phải bằng oracle `O(n^3)`.

In [27]:
rng = random.Random(713)
count = 0
for _ in range(1500):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 8) for _ in range(n)]
    k = rng.randint(0, 300)
    check_all(nums, k)
    count += 1
print("đã đối chiếu", count, "input ngẫu nhiên với", len(METHODS), "cách: OK")

đã đối chiếu 1500 input ngẫu nhiên với 5 cách: OK


In [28]:
# Lượt hai: giá trị và k theo đúng giới hạn đề, mảng vừa phải; oracle vẫn là tích đầy đủ (n <= 40)
rng = random.Random(7130)
for _ in range(300):
    n = rng.randint(1, 40)
    nums = [rng.choice([1, 1, 2, 3, 5, 10, 100, 1000, rng.randint(1, 1000)]) for _ in range(n)]
    k = rng.choice([0, 1, 2, 10 ** 6, rng.randint(0, 10 ** 6), rng.randint(1, 1000)])
    check_all(nums, k)
print("lượt 2 (giới hạn đề, n <= 40): OK")

lượt 2 (giới hạn đề, n <= 40): OK


Thêm lượt **mảng dài** (tới vài nghìn phần tử), đối chiếu giữa các cách nhanh với cách 1 (chính xác, dừng sớm) làm chuẩn; và **mảng có tích đúng bằng `k` rất nhiều lần** (lũy thừa của `10`) để thử độ chính xác của cách logarit:

In [29]:
rng = random.Random(1)
for _ in range(100):
    n = rng.randint(500, 3000)
    nums = [rng.choice([1, 1, 1, 2, 3, 10, rng.randint(1, 1000)]) for _ in range(n)]
    k = rng.randint(2, 10 ** 6)
    ref = METHODS["1-hai-vong"](nums, k)
    for name in ["2-cua-so", "3-log-nhi-phan", "4-chia-de-tri", "nop"]:
        assert METHODS[name](nums, k) == ref, (name, n, k)

for _ in range(100):                           # tích đúng bằng k: lũy thừa của 10 và các hệ số 2, 5
    n = rng.randint(50, 400)
    nums = [rng.choice([1, 2, 5, 10, 10, 100]) for _ in range(n)]
    k = rng.choice([10, 100, 1000, 10 ** 4, 10 ** 5, 10 ** 6, 20, 200, 2000, 50, 500, 5000])
    ref = METHODS["1-hai-vong"](nums, k)
    for name in ["2-cua-so", "3-log-nhi-phan", "4-chia-de-tri"]:
        assert METHODS[name](nums, k) == ref, (name, k)
print("mảng dài và tích đúng bằng k: OK")

mảng dài và tích đúng bằng k: OK


### 12.5. Hiệu năng ở giới hạn (`n = 3 * 10^4`)

Các trường hợp xấu cho từng cách: toàn `1` với `k = 10^6` (cửa sổ không bao giờ co: cách 1 xấu nhất, đáp án `n(n+1)/2`), toàn `1000` (cửa sổ luôn co), và ngẫu nhiên. Cách 1 **chỉ** chạy trên ngẫu nhiên (hai trường hợp đầu sẽ mất hàng chục giây).

In [30]:
N = 30000
rng_perf = random.Random(3)
cases = {
    "toàn 1, k=10^6": ([1] * N, 10 ** 6),
    "toàn 1000, k=10^6": ([1000] * N, 10 ** 6),
    "ngẫu nhiên 1..1000": ([rng_perf.randint(1, 1000) for _ in range(N)], 10 ** 6),
    "ngẫu nhiên 1..3": ([rng_perf.randint(1, 3) for _ in range(N)], 10 ** 6),
}
for label, (nums, k) in cases.items():
    row, results = [], set()
    for name in ["2-cua-so", "3-log-nhi-phan", "4-chia-de-tri", "nop"]:
        t0 = time.perf_counter()
        r = METHODS[name](nums, k)
        row.append(f"{name} {(time.perf_counter() - t0) * 1000:6.1f}ms")
        results.add(r)
    assert len(results) == 1
    print(f"{label:20} -> {results.pop():>10} | " + " | ".join(row))

assert METHODS["nop"]([1] * N, 10 ** 6) == N * (N + 1) // 2
assert METHODS["nop"]([1000] * N, 10 ** 6) == N
nums, k = cases["ngẫu nhiên 1..1000"]
t0 = time.perf_counter()
assert METHODS["1-hai-vong"](nums, k) == METHODS["nop"](nums, k)
print(f"cách 1 trên ngẫu nhiên 1..1000: {(time.perf_counter() - t0) * 1000:.1f} ms (đúng, vì dừng sớm)")

toàn 1, k=10^6       ->  450015000 | 2-cua-so    2.8ms | 3-log-nhi-phan    7.2ms | 4-chia-de-tri   50.3ms | nop    2.5ms
toàn 1000, k=10^6    ->      30000 | 2-cua-so    3.6ms | 3-log-nhi-phan    6.6ms | 4-chia-de-tri   20.0ms | nop    3.7ms
ngẫu nhiên 1..1000   ->      60880 | 2-cua-so    4.0ms | 3-log-nhi-phan    6.7ms | 4-chia-de-tri   24.7ms | nop   11.5ms


ngẫu nhiên 1..3      ->     691935 | 2-cua-so    5.1ms | 3-log-nhi-phan    7.0ms | 4-chia-de-tri   33.3ms | nop    7.6ms
cách 1 trên ngẫu nhiên 1..1000: 14.5 ms (đúng, vì dừng sớm)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `k <= 1`

**Code sai:**

In [31]:
def wrong_no_guard(nums, k):
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        while p >= k:                          # với k <= 1, p >= k luôn đúng
            p //= nums[l]
            l += 1                             # l vượt qua r + 1, rồi vượt cả chiều dài mảng
        ans += r - l + 1
    return ans

for nums, k in [([1, 2, 3], 0), ([1, 1], 1)]:
    try:
        print(nums, k, "sai:", wrong_no_guard(nums, k))
    except IndexError as e:
        print(nums, k, "sai: IndexError:", e)
    print(nums, k, "đúng:", Solution().numSubarrayProductLessThanK(nums, k))

[1, 2, 3] 0 sai: IndexError: list index out of range
[1, 2, 3] 0 đúng: 0
[1, 1] 1 sai: IndexError: list index out of range
[1, 1] 1 đúng: 0


**Vì sao sai:** khi `k <= 1`, ngay cả cửa sổ rỗng (`p = 1`) cũng thỏa `p >= k`, nên `while` không bao giờ dừng đúng lúc: `l` chạy quá cuối mảng và `nums[l]` báo lỗi (hoặc đáp án âm nếu không lỗi).

**Code đúng:** `if k <= 1: return 0` ở đầu hàm.

### Lỗi 2: Dùng `>` thay vì `>=` (đếm cả mảng con có tích bằng `k`)

In [32]:
def wrong_strict(nums, k):
    if k <= 1:
        return 0
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        while p > k:                           # SAI: phải là p >= k
            p //= nums[l]
            l += 1
        ans += r - l + 1
    return ans

print("sai  :", wrong_strict([10, 5, 2, 6], 100), "(tính cả [10,5,2] có tích đúng 100)")
print("đúng :", Solution().numSubarrayProductLessThanK([10, 5, 2, 6], 100))

sai  : 9 (tính cả [10,5,2] có tích đúng 100)
đúng : 8


**Vì sao sai:** đề yêu cầu tích **nhỏ hơn hẳn** `k`; mảng con có tích `== k` **không** hợp lệ, phải bị loại: thu hẹp khi `p >= k`.

**Code đúng:** `while p >= k:`.

### Lỗi 3: Đếm `r - l` thay vì `r - l + 1`

In [33]:
def wrong_off_by_one(nums, k):
    if k <= 1:
        return 0
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        while p >= k:
            p //= nums[l]
            l += 1
        ans += r - l                           # SAI: thiếu 1
    return ans

print("sai  :", wrong_off_by_one([10, 5, 2, 6], 100))
print("đúng :", Solution().numSubarrayProductLessThanK([10, 5, 2, 6], 100))

sai  : 4
đúng : 8


**Vì sao sai:** cửa sổ `[l..r]` có `r - l + 1` vị trí đầu trái khả dĩ (kể cả `l` và `r`). Ví dụ `l = r` (cửa sổ một phần tử): có đúng `1` mảng con, không phải `0`.

**Code đúng:** `ans += r - l + 1`.

### Lỗi 4: Dùng `if` thay vì `while` khi thu hẹp

In [34]:
def wrong_if(nums, k):
    if k <= 1:
        return 0
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        if p >= k:                             # SAI: có thể cần bỏ nhiều phần tử
            p //= nums[l]
            l += 1
        ans += r - l + 1
    return ans

print("sai  :", wrong_if([1, 1, 1, 100, 1], 50), "(không thu hẹp đủ khi gặp 100)")
print("đúng :", Solution().numSubarrayProductLessThanK([1, 1, 1, 100, 1], 50))
print("sai  :", wrong_if([10, 5, 2, 6], 20), "| đúng:", Solution().numSubarrayProductLessThanK([10, 5, 2, 6], 20))

sai  : 12 (không thu hẹp đủ khi gặp 100)
đúng : 7
sai  : 6 | đúng: 6


**Vì sao sai:** một phần tử mới có thể làm tích tăng rất nhiều (ví dụ nhân `100`), và phải bỏ **nhiều** phần tử bên trái (hoặc cả cửa sổ) mới đưa tích về dưới `k`. `if` chỉ bỏ một lần.

**Code đúng:** `while p >= k:`.

### Lỗi 5: Logarit không chừa biên an toàn

Với cách 3, dùng thẳng `T = log(k)` rồi so sánh `pre[j] - pre[i] < log(k)` có thể sai ở ranh giới do **sai số số thực**. Ta để máy **tìm** một phản ví dụ bằng cách thử nhiều mảng nhỏ:

In [35]:
def wrong_log(nums, k):
    if k <= 1:
        return 0
    pre = [0.0]
    for x in nums:
        pre.append(pre[-1] + log(x))
    T = log(k)                                 # SAI: không chừa biên cho tích đúng bằng k
    ans = 0
    for i in range(len(nums)):
        j = bisect_left(pre, pre[i] + T, i + 1)
        ans += j - i - 1
    return ans

rng = random.Random(713)
found = None
for _ in range(200000):
    nums = [rng.choice([2, 3, 4, 5, 6, 7, 8, 9, 10]) for _ in range(rng.randint(1, 6))]
    k = rng.choice([10, 100, 1000, 12, 24, 36, 60, 120, 720, 5040, 10 ** 4, 10 ** 5, 10 ** 6])
    if wrong_log(nums, k) != Solution().numSubarrayProductLessThanK(nums, k):
        found = (nums, k)
        break

nums, k = found
print("phản ví dụ:", nums, "k =", k)
print("log thô  :", wrong_log(nums, k))
print("đúng     :", Solution().numSubarrayProductLessThanK(nums, k), "| cách 3 (log(k - 0.5)):", Solution3().numSubarrayProductLessThanK(nums, k))
pre = [0.0]
for x in nums:
    pre.append(pre[-1] + log(x))
print("log(2*4*3) =", log(24), "vs tổng log =", log(2) + log(4) + log(3), "-> khác nhau ở chữ số cuối, so sánh < sai")

phản ví dụ: [2, 4, 3] k = 12
log thô  : 5
đúng     : 4 | cách 3 (log(k - 0.5)): 4
log(2*4*3) = 3.1780538303479458 vs tổng log = 3.1780538303479453 -> khác nhau ở chữ số cuối, so sánh < sai


**Vì sao sai:** tích `2*4*3 = 12 = k` (không hợp lệ), nhưng `log(2) + log(4) + log(3)` có thể **hơi nhỏ hơn** `log(12)` do làm tròn, nên phép so sánh `<` coi nó hợp lệ.

**Code đúng:** vì tích là số nguyên, `tích < k` <=> `tích < k - 0.5`: dùng `T = log(k - 0.5)`, khoảng đệm `>= 5 * 10^-7` lớn hơn nhiều sai số cộng dồn; hoặc tránh logarit và dùng cửa sổ trượt với số nguyên (cách 2).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "đếm mảng con liên tiếp thỏa điều kiện trên **tổng/tích** với phần tử **dương**" -> **cửa sổ trượt**. Điều kiện phải **đơn điệu**: nới cửa sổ chỉ làm giá trị tăng, thu hẹp chỉ làm giảm.
- **Công thức đếm:** với mỗi đuôi `r`, số mảng con hợp lệ kết thúc tại `r` là `r - l + 1`.
- **Bảo vệ biên:** kiểm tra `k <= 1` (và mọi trường hợp cửa sổ có thể rỗng).
- **`>=` và `<`:** "nhỏ hơn hẳn" -> thu hẹp khi `p >= k`.
- **Dùng `//`** để bỏ phần tử khỏi tích (chia hết); không dùng `/`.
- **Khi có số 0 hoặc số âm** tính đơn điệu mất; cửa sổ trượt không áp dụng trực tiếp (xem biến thể 15.3).
- **Tích -> tổng:** `log` biến tích thành tổng nhưng đừng so sánh số thực tại ranh giới; chừa biên bằng `k - 0.5`.
- Bài họ hàng: LeetCode 209 (Minimum Size Subarray Sum), 3 (Longest Substring Without Repeating Characters), 904 (Fruit Into Baskets), 930 (Binary Subarrays With Sum), 992 (Subarrays with K Different Integers), 152 (Maximum Product Subarray), 560 (Subarray Sum Equals K).

## 15. Biến thể

### 15.1. Tích nhỏ hơn **hoặc bằng** `k`

Đổi điều kiện thu hẹp thành `p > k`; với `k < 1` đáp án `0` (mọi tích `>= 1`). Chú ý `k = 1`: chỉ mảng con toàn số `1` có tích `1 <= 1`.

In [36]:
def count_at_most(nums, k):
    if k < 1:
        return 0
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        while p > k:
            p //= nums[l]
            l += 1
        ans += r - l + 1
    return ans

rng = random.Random(151)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 100)
    n = len(nums)
    assert count_at_most(nums, k) == sum(1 for i in range(n) for j in range(i, n) if prod(nums[i:j + 1]) <= k)
print(count_at_most([10, 5, 2, 6], 100), "(tính cả [10,5,2] = 100)")

9 (tính cả [10,5,2] = 100)


### 15.2. Mảng con dài nhất có tích nhỏ hơn `k`

Cùng cửa sổ trượt, nhưng thay vì đếm ta theo dõi độ dài lớn nhất `r - l + 1`.

In [37]:
def longest_product_less_than(nums, k):
    if k <= 1:
        return 0
    p, l, best = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        while p >= k:
            p //= nums[l]
            l += 1
        best = max(best, r - l + 1)
    return best

print(longest_product_less_than([10, 5, 2, 6], 100), longest_product_less_than([1, 1, 1, 100], 50))
rng = random.Random(152)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 100)
    n = len(nums)
    exp = max((j - i + 1 for i in range(n) for j in range(i, n) if prod(nums[i:j + 1]) < k), default=0)
    assert longest_product_less_than(nums, k) == exp
print("15.2 khớp vét cạn")

3 3
15.2 khớp vét cạn


### 15.3. Cho phép số `0` (mảng `nums[i] >= 0`)

Tính đơn điệu bị phá: tích chứa `0` bằng `0 < k` (với `k > 0`). Ý tưởng: **mọi mảng con chứa ít nhất một số `0`** đều hợp lệ (khi `k > 0`); các mảng con không chứa `0` thuộc các **khúc** giữa các số `0`, trong từng khúc ta dùng cửa sổ trượt như cũ. Đáp án `= (tổng số mảng con) - (số mảng con không chứa 0) + (số mảng con không chứa 0 và hợp lệ)`.

In [38]:
def count_with_zeros(nums, k):
    if k <= 0:
        return 0
    n = len(nums)
    total = n * (n + 1) // 2
    # mỗi khúc không chứa 0: bớt các mảng con của khúc, cộng lại số mảng hợp lệ
    ans = total
    i = 0
    while i < n:
        if nums[i] == 0:
            i += 1
            continue
        j = i
        while j < n and nums[j] != 0:
            j += 1
        seg = nums[i:j]
        m = len(seg)
        ans -= m * (m + 1) // 2
        ans += Solution().numSubarrayProductLessThanK(seg, k)   # k >= 1 ở đây; k == 1 trả 0 (đúng: tích >= 1)
        i = j
    return ans

rng = random.Random(153)
for _ in range(500):
    nums = [rng.choice([0, 0, 1, 2, 3, 4, 5]) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 40)
    n = len(nums)
    assert count_with_zeros(nums, k) == sum(1 for i in range(n) for j in range(i, n) if prod(nums[i:j + 1]) < k)
print(count_with_zeros([0, 2, 3], 5), "-> [0],[0,2],[0,2,3],[2],[3] = 5 (còn [2,3]=6 không)")

5 -> [0],[0,2],[0,2,3],[2],[3] = 5 (còn [2,3]=6 không)


### 15.4. Liệt kê các mảng con hợp lệ (chỉ với dữ liệu nhỏ)

Với `k` nhỏ có thể liệt kê từng mảng con bằng một **generator** dựa trên cửa sổ trượt: ứng với mỗi `r`, các mảng con là `nums[start:r+1]` cho `start` từ `l` đến `r`. Chú ý: số lượng có thể rất lớn, chỉ dùng cho dữ liệu nhỏ.

In [39]:
def list_subarrays(nums, k):
    if k <= 1:
        return
    p, l = 1, 0
    for r, x in enumerate(nums):
        p *= x
        while p >= k:
            p //= nums[l]
            l += 1
        for start in range(l, r + 1):
            yield nums[start:r + 1]

out = list(list_subarrays([10, 5, 2, 6], 100))
print(len(out), out)
assert len(out) == 8

8 [[10], [10, 5], [5], [5, 2], [2], [5, 2, 6], [2, 6], [6]]


### 15.5. Tổng nhỏ hơn `k` (cùng khuôn mẫu)

Thay "tích" bằng "tổng" (với `nums[i] >= 1`): cộng thay cho nhân, trừ thay cho chia. Cùng tính đơn điệu, cùng công thức đếm `r - l + 1`.

In [40]:
def count_sum_less_than(nums, k):
    s, l, ans = 0, 0, 0
    for r, x in enumerate(nums):
        s += x
        while s >= k and l <= r:
            s -= nums[l]
            l += 1
        ans += r - l + 1
    return ans

rng = random.Random(155)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 30)
    n = len(nums)
    assert count_sum_less_than(nums, k) == sum(1 for i in range(n) for j in range(i, n) if sum(nums[i:j + 1]) < k)
print(count_sum_less_than([10, 5, 2, 6], 15))

7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp | duyệt mọi mảng con, dừng sớm khi `p >= k` | `O(n^2)` xấu nhất | `O(1)` | `n` nhỏ, làm oracle nhanh |
| 2. Cửa sổ trượt | hai con trỏ không lùi; `r - l + 1` mảng/đuôi | `O(n)` | `O(1)` | **nộp bài** |
| 3. Log + nhị phân | tích -> tổng log, `bisect_left` tìm đuôi | `O(n log n)` | `O(n)` | học kỹ thuật tích -> tổng; cần chừa biên |
| 4. Chia để trị | đếm mảng con cắt qua điểm giữa bằng hai con trỏ | `O(n log n)` | `O(n)` tạm | khi cửa sổ trượt không dùng được |

Kết luận: nộp **cách 2**. Cách 1 là chuẩn đối chiếu đơn giản; cách 3 và 4 mở rộng góc nhìn (prefix sum + tìm kiếm nhị phân, chia để trị).

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra rồi chạy:

In [41]:
print(Solution().numSubarrayProductLessThanK([1, 2, 3], 0))
print(Solution().numSubarrayProductLessThanK([1, 1, 1], 2))
print(Solution().numSubarrayProductLessThanK([10, 5, 2, 6], 100))

0
6
8


**Bài 2 (tính tay).** `nums = [1, 2, 3]`, `k = 4`. Liệt kê 6 mảng con và tích, đếm số mảng có tích `< 4`, gán vào `ans2` (đang là `None`).

In [42]:
ans2 = None      # TODO: điền một số nguyên

**Bài 3 (điền TODO).** Viết `windows_ending_at(nums, k, r)` trả về số mảng con **kết thúc tại chỉ số `r`** có tích `< k`, bằng cách đi ngược từ `r` về trái, giữ tích chạy và dừng khi `>= k`. Ví dụ với `[10,5,2,6]`, `k = 100`: `r = 2` cho `2`, `r = 3` cho `3`.

In [43]:
def windows_ending_at(nums, k, r):
    # TODO
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_count(nums, k)` theo cách 2 (cửa sổ trượt, nhớ chặn `k <= 1`).

In [44]:
def my_count(nums, k):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `count_at_most_k(nums, k)` đếm các mảng con có tích **nhỏ hơn hoặc bằng** `k` (mục 15.1). (b) Viết `exact_k_count(nums, k)` đếm các mảng con có tích **đúng bằng** `k`. Gợi ý: `exact(k) = at_most(k) - at_most(k - 1)` (tích là số nguyên).

In [45]:
def count_at_most_k(nums, k):
    # TODO
    pass

def exact_k_count(nums, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `k = 0` -> `0`; mảng toàn `1` với `k = 2`: mọi mảng con đều hợp lệ.
- Bài 2: các mảng con: `[1]`, `[2]`, `[3]`, `[1,2]`, `[2,3]`, `[1,2,3]`; tính tích từng mảng.
- Bài 3: duyệt `i` từ `r` giảm dần; nhân dần; dừng khi tích `>= k`.
- Bài 4: hai con trỏ `l`, `r`; thu hẹp khi tích `>= k`.
- Bài 5: nhớ rằng bài hiện tại thu hẹp khi `p >= k`; với "nhỏ hơn hoặc bằng" thì thu hẹp khi `p > k`.

**Mức 2 (vừa):**
- Bài 2: tích lần lượt `1, 2, 3, 2, 6, 6`; đếm những số `< 4`.
- Bài 3: `p = 1; count = 0; for i in range(r, -1, -1): p *= nums[i]; if p >= k: break; count += 1`.
- Bài 4: `while p >= k: p //= nums[l]; l += 1`, rồi `ans += r - l + 1`.
- Bài 5: `at_most(k)` là bài gốc với `k + 1` (vì tích nguyên: `<= k` tương đương `< k + 1`).

**Mức 3 (gần đáp án):**
- Bài 2: bốn mảng: `[1]`, `[2]`, `[3]`, `[1,2]`.
- Bài 5(a): `return Solution().numSubarrayProductLessThanK(nums, k + 1)`.
- Bài 5(b): `at_most(k) - at_most(k - 1)`; chú ý `k < 1` thì `0`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [46]:
# Bài 1: 0, 6, 8
assert Solution().numSubarrayProductLessThanK([1, 2, 3], 0) == 0
assert Solution().numSubarrayProductLessThanK([1, 1, 1], 2) == 6
assert Solution().numSubarrayProductLessThanK([10, 5, 2, 6], 100) == 8

# Bài 2: tích 1, 2, 3, 2, 6, 6 -> bốn mảng có tích < 4
ans2 = 4
assert ans2 == Solution().numSubarrayProductLessThanK([1, 2, 3], 4) == oracle_prod([1, 2, 3], 4)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [47]:
# Bài 3
def windows_ending_at(nums, k, r):
    p, count = 1, 0
    for i in range(r, -1, -1):
        p *= nums[i]
        if p >= k:
            break
        count += 1
    return count

nums = [10, 5, 2, 6]
assert [windows_ending_at(nums, 100, r) for r in range(4)] == [1, 2, 2, 3]
assert sum(windows_ending_at(nums, 100, r) for r in range(4)) == 8
rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    k = rng.randint(0, 100)
    assert sum(windows_ending_at(nums, k, r) for r in range(len(nums))) == oracle_prod(nums, k)
print("Bài 3 OK")

Bài 3 OK


In [48]:
# Bài 4
def my_count(nums, k):
    if k <= 1:
        return 0
    p, l, ans = 1, 0, 0
    for r, x in enumerate(nums):
        p *= x
        while p >= k:
            p //= nums[l]
            l += 1
        ans += r - l + 1
    return ans

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(1, 8) for _ in range(rng.randint(1, 12))]
    k = rng.randint(0, 300)
    assert my_count(nums, k) == oracle_prod(nums, k)
assert my_count([1] * 100, 10 ** 6) == 100 * 101 // 2
print("Bài 4 OK")

Bài 4 OK


In [49]:
# Bài 5
def count_at_most_k(nums, k):
    if k < 1:
        return 0
    return Solution().numSubarrayProductLessThanK(nums, k + 1)

def exact_k_count(nums, k):
    return count_at_most_k(nums, k) - count_at_most_k(nums, k - 1)

assert count_at_most_k([10, 5, 2, 6], 100) == 9
assert exact_k_count([10, 5, 2, 6], 100) == 1                  # chỉ [10,5,2]
rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    k = rng.randint(0, 80)
    n = len(nums)
    assert count_at_most_k(nums, k) == sum(1 for i in range(n) for j in range(i, n) if prod(nums[i:j + 1]) <= k)
    assert exact_k_count(nums, k) == sum(1 for i in range(n) for j in range(i, n) if prod(nums[i:j + 1]) == k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template cửa sổ trượt (tích dương, "nhỏ hơn hẳn" `k`):**

```python
class Solution:
    def numSubarrayProductLessThanK(self, nums: List[int], k: int) -> int:
        if k <= 1:                       # mọi tích >= 1 nên không có mảng con nào
            return 0
        p, l, ans = 1, 0, 0
        for r, x in enumerate(nums):
            p *= x                       # mở rộng sang phải
            while p >= k:                # thu hẹp bên trái cho tới khi p < k
                p //= nums[l]
                l += 1
            ans += r - l + 1             # số mảng con kết thúc tại r
        return ans
```

**Mẹo đếm "đúng bằng" / "nhỏ hơn hoặc bằng":** `<= k` = bài gốc với `k + 1` (tích nguyên); `== k` = `at_most(k) - at_most(k - 1)`.

**Checklist trước khi submit:**

- [ ] Đã chặn `k <= 1` (trả `0`)?
- [ ] Thu hẹp bằng **`while p >= k`** (không phải `>` và không phải `if`)?
- [ ] Cộng **`r - l + 1`** (không phải `r - l`)?
- [ ] Bỏ phần tử bằng **`p //= nums[l]`** (chia nguyên, không `/`)?
- [ ] Không dùng logarit nếu không chừa biên (`log(k - 0.5)`)?
- [ ] Đã thử: `k = 0`, `k = 1`, mảng toàn `1`, tích đúng bằng `k`, phần tử đơn lẻ `>= k`?
- [ ] Hiểu vì sao cần `nums[i] >= 1` (tính đơn điệu); nếu có `0` thì xử lý riêng?